In [1]:
	from ortools.sat.python import cp_model
	def get_min_edges(n, k, d=0, lower_bound=0):
	    # 最大可能三角形数为 C(n, 3)，超过直接判定无解
	    max_triangles = n * (n - 1) * (n - 2) // 6
	    if k > max_triangles:
	        return None
	    model = cp_model.CpModel()
	    # 1. 定义边变量 (只定义 i < j 的上三角)
	    x = {}
	    for i in range(n):
	        for j in range(i+1, n):
	            x[i, j] = model.NewBoolVar(f'x_{i}_{j}')
	    # 2. 固定特定的子图结构 (对称性破缺)
	    # 中心点0连向1,2,3,4
	    model.Add(x[0, 1] == 1)
	    model.Add(x[0, 2] == 1)
	    model.Add(x[0, 3] == 1)
	    model.Add(x[0, 4] == 1)
	    # 两条独立边 5-6, 7-8
	    model.Add(x[5, 6] == 1)
	    model.Add(x[7, 8] == 1)
	    # 3. 度数约束 (针对第三问)
	    if d > 0:
	        for i in range(n):
	            deg = []
	            for j in range(n):
	                if i < j:
	                    deg.append(x[i, j])
	                elif i > j:
	                    deg.append(x[j, i])
	            # 修复了语法错误
	            model.Add(sum(deg) >= d)
	    # 4. 三角形约束 (修复逻辑错误：必须双向约束，保证当且仅当三边均存在时t=1)
	    t = []
	    for i in range(n):
	        for j in range(i+1, n):
	            for l in range(j+1, n):
	                t_var = model.NewBoolVar(f't_{i}_{j}_{l}')
	                # 如果 t=1，则三条边必须都为 1
	                model.AddBoolAnd([x[i, j], x[i, l], x[j, l]]).OnlyEnforceIf(t_var)
	                # 如果 t=0，则至少有一条边为 0 (防止求解器作弊将t强行置0)
	                model.AddBoolOr([x[i, j].Not(), x[i, l].Not(), x[j, l].Not()]).OnlyEnforceIf(t_var.Not())
	                t.append(t_var)
	    model.Add(sum(t) >= k)
	    # 5. 目标函数与下界注入
	    model.Minimize(sum(x.values()))
	    model.Add(sum(x.values()) >= lower_bound)
	    # 6. 求解
	    solver = cp_model.CpSolver()
	    solver.parameters.num_search_workers = 8
	    solver.parameters.max_time_in_seconds = 10.0
	    status = solver.Solve(model)
	    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
	        return int(solver.ObjectiveValue())
	    else:
	        return None
	# ==========================================
	# 测试与输出结果
	# ==========================================
	print("=== 1. 计算 E(n, k) (9 <= n <= 12) ===")
	for n in range(9, 13):
	    print(f"\n--- n = {n} ---")
	    prev_edges = 6  # 基础子图至少6条边
	    k = 0
	    while True:
	        res = get_min_edges(n, k, d=0, lower_bound=prev_edges)
	        if res is None:
	            print(f"k={k} 时已无解（达到最大三角形数）。")
	            break
	        print(f"E({n}, {k}) = {res}")
	        prev_edges = res
	        k += 1
	        # 达到完全图时提前终止，避免无效运算
	        if res == n * (n - 1) // 2:
	            print(f"已达到完全图，最大三角形数 k={n*(n-1)*(n-2)//6}。")
	            break
	print("\n\n=== 2. 计算 E(n, k, d) ===")
	# 取 n=9,10 且 d=2,3,4 为例
	for n in [9, 10]:
	    for d in [2, 3, 4]:
	        print(f"\n--- n = {n}, d = {d} ---")
	        # 利用握手定理计算下界：边数 >= n * d / 2
	        base_edges = max(6, (n * d + 1) // 2)
	        prev_edges = base_edges
	        k = 0
	        while True:
	            res = get_min_edges(n, k, d=d, lower_bound=prev_edges)
	            if res is None:
	                print(f"k={k} 时已无解。")
	                break
	            print(f"E({n}, {k}, {d}) = {res}")
	            prev_edges = res
	            k += 1
	            if res == n * (n - 1) // 2:
	                print(f"已达到完全图，最大三角形数 k={n*(n-1)*(n-2)//6}。")
	                break

=== 1. 计算 E(n, k) (9 <= n <= 12) ===

--- n = 9 ---
E(9, 0) = 6
E(9, 1) = 7
E(9, 2) = 8
E(9, 3) = 9
E(9, 4) = 9
E(9, 5) = 10
E(9, 6) = 11
E(9, 7) = 11
E(9, 8) = 12
E(9, 9) = 12
E(9, 10) = 12
E(9, 11) = 14
E(9, 12) = 15
E(9, 13) = 15
E(9, 14) = 16
E(9, 15) = 16
E(9, 16) = 16
E(9, 17) = 17
E(9, 18) = 17
E(9, 19) = 17
E(9, 20) = 17
E(9, 21) = 18
E(9, 22) = 19
E(9, 23) = 19
E(9, 24) = 20
E(9, 25) = 20
E(9, 26) = 20
E(9, 27) = 21
E(9, 28) = 21
E(9, 29) = 21
E(9, 30) = 21
E(9, 31) = 22
E(9, 32) = 22
E(9, 33) = 22
E(9, 34) = 22
E(9, 35) = 22
E(9, 36) = 24
E(9, 37) = 25
E(9, 38) = 25
E(9, 39) = 26
E(9, 40) = 26
E(9, 41) = 26
E(9, 42) = 27
E(9, 43) = 27
E(9, 44) = 27
E(9, 45) = 27
E(9, 46) = 28
E(9, 47) = 28
E(9, 48) = 28
E(9, 49) = 28
E(9, 50) = 28
E(9, 51) = 29
E(9, 52) = 29
E(9, 53) = 29
E(9, 54) = 29
E(9, 55) = 29
E(9, 56) = 29
E(9, 57) = 30
E(9, 58) = 31
E(9, 59) = 31
E(9, 60) = 32
E(9, 61) = 32
E(9, 62) = 32
E(9, 63) = 33
E(9, 64) = 33
E(9, 65) = 33
E(9, 66) = 33
E(9, 67) = 34
E(9, 68) = 